In [46]:
import pandas as pd 
import numpy as np 
import matplotlib as plt
import seaborn as sns
import warnings
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import mode
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans

## Cargar datos limpios para el entrenamiento

In [47]:
df = pd.read_csv("cardio_data_cleaned.csv")
df.head()

,gender,height,weight,ap_hi,ap_lo,smoke,alco,active,cardio,age_years,bmi,bp_category_encoded,cholesterol_1,cholesterol_2,cholesterol_3,gluc_1,gluc_2,gluc_3
0,1,0.470171,-0.873451,-1.046104,-0.101094,0,0,1,0,-0.414547,-1.097985,Hypertension Stage 1,1,0,0,1,0,0
1,0,-1.097398,0.894889,0.958655,1.053057,0,0,1,1,0.323978,1.634034,Hypertension Stage 2,0,0,1,1,0,0
2,0,0.078279,-0.719682,0.290402,-1.255244,0,0,0,1,-0.266842,-0.773217,Hypertension Stage 1,0,0,1,1,0,0
3,1,0.600802,0.664236,1.626907,2.207207,0,0,1,1,-0.709957,0.323480,Hypertension Stage 2,1,0,0,1,0,0
4,0,-1.097398,-1.334757,-1.714357,-2.409395,0,0,0,0,-0.857662,-0.877904,Normal,1,0,0,1,0,0


## Dividir los datos en conjuntos de entrenamiento y prueba

In [48]:
# Usar variables numéricas 
df_numeric = df.select_dtypes(include=[np.number])

# Separar features y target
X = df_numeric.drop(['cardio'], axis=1)
y = df_numeric['cardio']

# Dividir en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Crear modelos

In [49]:
modelos = {
    "Regresión Logística": LogisticRegression(random_state=42),
    "Árbol de Decisión": DecisionTreeClassifier(random_state=42, max_depth=10),
    "SVM": SVC(random_state=42, kernel='rbf', max_iter=2000),
}

# Guardar resultados
resultados = {}

## Entrenar y evaluar modelos 

In [50]:
# Ocultar advertencias de convergencia y de TensorFlow para una salida limpia
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

for name, model in modelos.items():
    print(f"  - Entrenando {name}...")
    # Entrenar
    model.fit(X_train, y_train)
    # Predecir
    y_pred = model.predict(X_test)
    
    # Calcular métricas
    resultados[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precisión": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    }

# Entrenar red neuronal
print("  - Entrenando Red Neuronal...")
# Construir la red neuronal con TensorFlow
nn_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') # Sigmoide porque es clasificación binaria 
])

# Compilar el modelo
nn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Entrenar el modelo
nn_model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.1, verbose=0)

# Predecir con TensorFlow 
# Todo lo que sea > 0.5 lo convertimos a clase 1, y lo demás a 0.
y_pred_prob_nn = nn_model.predict(X_test, verbose=0)
y_pred_nn = (y_pred_prob_nn > 0.5).astype(int).flatten()

# Guardar métricas de la Red Neuronal
resultados["Red Neuronal"] = {
    "Accuracy": accuracy_score(y_test, y_pred_nn),
    "Precisión": precision_score(y_test, y_pred_nn),
    "Recall": recall_score(y_test, y_pred_nn),
    "F1-Score": f1_score(y_test, y_pred_nn)
}


print("  - Entrenando K-Means (No supervisado)...")
# Entrenar K-Means 
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters_pred = kmeans.fit_predict(X_test)

# Mapear los clústeres a las etiquetas reales (0 o 1) basándose en la moda
y_pred_kmeans = np.zeros_like(clusters_pred)
for i in range(2):
    mask = (clusters_pred == i)
    if np.any(mask):
        # Asignar la etiqueta más común en este clúster
        y_pred_kmeans[mask] = mode(y_test.values[mask], keepdims=True)[0][0]

resultados["K-Means (Clustering)"] = {
    "Accuracy": accuracy_score(y_test, y_pred_kmeans),
    "Precisión": precision_score(y_test, y_pred_kmeans, zero_division=0),
    "Recall": recall_score(y_test, y_pred_kmeans),
    "F1-Score": f1_score(y_test, y_pred_kmeans)
}

# Crear DataFrame de resultados para comparar modelos
resultados_df = pd.DataFrame(resultados).T

  - Entrenando Regresión Logística...
  - Entrenando Árbol de Decisión...
  - Entrenando SVM...
  - Entrenando Red Neuronal...
  - Entrenando K-Means (No supervisado)...


## Comparar resultados de cada modelo

In [51]:
display(resultados_df.sort_values(by="Accuracy", ascending=False))

,Accuracy,Precisión,Recall,F1-Score
Red Neuronal,0.728323,0.751953,0.658120,0.701914
Regresión Logística,0.726208,0.749113,0.656566,0.699793
Árbol de Decisión,0.723943,0.743603,0.659363,0.698954
K-Means (Clustering),0.665710,0.694935,0.556488,0.618053
SVM,0.481873,0.470375,0.524320,0.495885


In [ ]:
# Configurar el estilo visual de los gráficos
sns.set_theme(style="whitegrid")

# =====================================================================
# 1. GRÁFICO DE BARRAS AGRUPADAS (Todas las métricas por modelo)
# =====================================================================
# Utilizamos el método .plot() integrado en pandas para hacer barras agrupadas
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(14, 8))

# El colormap 'viridis' es amigable a la vista y daltónicos
resultados_df.plot(kind='bar', ax=ax, colormap='viridis', edgecolor='black', width=0.8)

plt.title('Comparación de Todas las Métricas por Modelo', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Modelos de Machine Learning', fontsize=12)
plt.ylabel('Puntuación (0.0 a 1.0)', fontsize=12)

# Rotar las etiquetas del eje X para que los nombres de los modelos se lean bien
plt.xticks(rotation=45, ha='right', fontsize=11)

# Ajustar la leyenda para que no tape las barras
plt.legend(title='Métricas', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10)
plt.ylim(0, 1.05) # Límite del eje Y de 0 a 1 (100%)

# tight_layout ajusta automáticamente los márgenes para que nada quede cortado
plt.tight_layout()
plt.show()


# =====================================================================
# 2. MAPA DE CALOR (Heatmap) para leer los valores numéricos exactos
# =====================================================================
plt.figure(figsize=(10, 6))

# Creamos el mapa de calor pasando nuestro DataFrame
# annot=True muestra los números, fmt='.3f' muestra 3 decimales
sns.heatmap(resultados_df, annot=True, cmap='YlGnBu', fmt='.3f', linewidths=.5, cbar_kws={'label': 'Puntuación'})

plt.title('Mapa de Calor: Rendimiento de Modelos', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Métricas', fontsize=12)
plt.ylabel('Modelos', fontsize=12)

plt.xticks(fontsize=10)
plt.yticks(fontsize=11, rotation=0)

plt.tight_layout()
plt.show()

AttributeError: module 'matplotlib' has no attribute 'subplots'